# 02 — Well-Log Analysis

Temperature gradient computation, thermal anomaly detection, and prospect ranking.

**Objectives:**
- Compute linear temperature gradients for all wells
- Detect anomalously high-gradient zones
- Rank prospects by geothermal potential
- Estimate conductive heat flow

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.well_analysis import TemperatureGradient, WellProfile

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 2.1 Load Clean Data

In [ ]:
data = pd.read_csv('../data/sample_wells.csv')
data = data.dropna(subset=['temperature_c'])
print(f'Loaded {len(data)} measurements for {data["well_id"].nunique()} wells')

## 2.2 Temperature Gradient Analysis

In [ ]:
# Compute gradients for all wells
tg = TemperatureGradient(data)
results = tg.compute_all()

print('Temperature Gradient Results (sorted by potential):')
print(results.sort_values('gradient_C_per_km', ascending=False)
      [['rank', 'well_id', 'gradient_C_per_km', 'max_temp_C', 'max_depth_m', 'r_squared']]
      .to_string(index=False))

## 2.3 Gradient Visualization

In [ ]:
# Bar chart of gradients
plot_data = results.dropna(subset=['gradient_C_per_km']).sort_values('gradient_C_per_km')

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.YlOrRd(np.linspace(0.2, 1.0, len(plot_data)))

ax.barh(plot_data['well_id'], plot_data['gradient_C_per_km'], color=colors)
ax.axvline(x=60, color='red', linestyle='--', alpha=0.7, label='High-grade threshold (60 °C/km)')
ax.axvline(x=30, color='orange', linestyle='--', alpha=0.5, label='Normal threshold (30 °C/km)')
ax.set_xlabel('Temperature Gradient (°C/km)', fontsize=12)
ax.set_ylabel('Well ID', fontsize=12)
ax.set_title('Geothermal Prospects Ranked by Temperature Gradient', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../images/02_gradient_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Thermal Anomaly Detection

In [ ]:
# Detect wells with anomalously high gradients
anomalies = tg.detect_anomalies(threshold_percentile=75)
print('\nAnomaly Detection Results:')
print(anomalies[['well_id', 'gradient_C_per_km', 'classification', 'is_anomaly']].to_string(index=False))

print(f'\nAnomalous wells: {anomalies["is_anomaly"].sum()} / {len(anomalies)}')

In [ ]:
# Classification distribution
fig, ax = plt.subplots(figsize=(8, 4))
anomalies['classification'].value_counts().plot(kind='pie', ax=ax, autopct='%1.0f%%',
    colors=['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad'])
ax.set_ylabel('')
ax.set_title('Geothermal Gradient Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/02_classification_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Heat Flow Estimation

In [ ]:
# Estimate conductive heat flow (assuming k = 2.5 W/(m·K))
heat_flow = tg.compute_heat_flow(thermal_conductivity=2.5)
print('Heat Flow Estimates (top 10):')
print(heat_flow.nlargest(10, 'heat_flow_mW_m2')
      [['well_id', 'gradient_C_per_km', 'heat_flow_mW_m2']]
      .to_string(index=False))

## 2.6 Depth-Temperature Profiles

In [ ]:
# Plot temperature profiles for top 5 wells
top_wells = results.nlargest(5, 'gradient_C_per_km')['well_id'].tolist()

fig, ax = plt.subplots(figsize=(8, 10))
for well in top_wells:
    well_data = data[data['well_id'] == well].sort_values('depth_m')
    ax.plot(well_data['temperature_c'], well_data['depth_m'], 'o-', label=well, markersize=5)

ax.invert_yaxis()
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Depth (m)', fontsize=12)
ax.set_title('Temperature vs Depth — Top 5 Prospects', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/02_temp_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Next:** [03 — Geospatial Visualization](03_geospatial_visualization.ipynb)